In [0]:
%run "../../utils/00_config"

In [0]:
%run "../../utils/02_data_quality"

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

stg_catalog = spark.table(
    get_table("staging", "stg_wfm_catalog")
)

print(f"Source rows: {stg_catalog.count()}")

In [0]:
window_latest = (
    Window
    .partitionBy("item_id")
    .orderBy(
        F.to_timestamp("observed_at").desc(),
        F.col("execution_id").desc()
    )
)

dim_item = (
    stg_catalog
    .withColumn("_row_number", F.row_number().over(window_latest))
    .filter(F.col("_row_number") == 1)
    .drop("_row_number")
)

In [0]:
dim_item = dim_item.select(
    "item_id",
    "item_slug",
    "item_name",
    "product_slug",
    "category",
    "game_ref",
    "tags",
    "is_set_root",
    "set_part_ids",
    "quantity_in_set",
    "ducats",
    "required_mastery_rank",
    "trading_tax",
    "is_tradable",
    "description",
    "wiki_url",
    "icon_path",
    "thumbnail_path",
    "sub_icon_path",
    F.col("execution_id").alias("source_execution_id"),
    F.col("observed_at").alias("catalog_observed_at")
)

In [0]:
check_required_fields(
    dim_item,
    ["item_id", "item_slug", "item_name", "product_slug", "category"]
)

check_unique_keys(dim_item, ["item_id"])

print(f"Dimension rows: {dim_item.count()}")

display(
    dim_item.orderBy("product_slug", "is_set_root", "item_slug")
)

In [0]:
from delta.tables import DeltaTable

target_table = get_table("intermediate", "dim_item")

if spark.catalog.tableExists(target_table):
    delta_table = DeltaTable.forName(spark, target_table)

    merge_condition = "target.item_id = source.item_id"

    (
        delta_table.alias("target")
        .merge(dim_item.alias("source"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
else:
    (
        dim_item.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(target_table)
    )

print(f"Table materialized: {target_table}")

In [0]:
dim_item_table = spark.table(target_table)

check_unique_keys(dim_item_table, ["item_id"])

print(f"Total records: {dim_item_table.count()}")

display(
    dim_item_table
    .orderBy("product_slug", "is_set_root", "item_slug")
)